# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Display name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
from pprint import pprint

# List all record sets with their @id and name
print("Available Record Sets:")
record_sets = {rs['@id']: rs for rs in metadata.to_json().get('recordSet', [])}
for rec_id, rec in record_sets.items():
    print(f"- @id: {rec_id}, Name: {rec.get('name', '')}")
    # List the columns/fields for each record set
    columns = rec.get('field', [])
    if isinstance(columns, dict):  # Some may be a dict if only one column
        columns = [columns]
    print("  Fields:")
    for c in columns:
        # ensure field is resolved (might be an @id reference)
        if isinstance(c, dict):
            c_id = c.get('@id', c)
        else:
            c_id = c
        print(f"    - {c_id}")
print("\nExample records for each record set:")
for record_set_id in record_sets:
    print(f"\nRecord set {record_set_id}:")
    try:
        for idx, record in enumerate(dataset.records(record_set=record_set_id)):
            pprint(record)
            if idx > 1:
                break
    except Exception as e:
        print(f"  Failed to load records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Find all record set IDs
record_set_ids = list(record_sets.keys())

# Load all record sets into DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Record set {record_set_id}: error - {e}")

if record_set_ids:
    print(f"Fields in record set {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick the first record set (as example)
if record_set_ids:
    example_record_set = record_set_ids[0]
    df = dataframes[example_record_set]
    # Identify numeric fields by checking type of a few non-null entries
    numeric_fields = [
        col for col in df.columns 
        if pd.api.types.is_numeric_dtype(df[col]) or pd.to_numeric(df[col], errors='coerce').notna().sum() > 0
    ]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field: {numeric_field}")
        # Remove non-numeric values
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by another field
        other_fields = [col for col in df.columns if col != numeric_field]
        group_field = None
        for f in other_fields:
            # Choose grouping field with less than 10 unique values
            if df[f].nunique() > 1 and df[f].nunique() < 10:
                group_field = f
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No record sets found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualize the numeric field distribution
if record_set_ids and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {example_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # If grouped_df is available, show mean per group field
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.